In [50]:
import pandas as pd 
import glob
import os

In [28]:
df_2025 = pd.read_excel('Unleaded_Retail (Incl. Tax)_WEEKLY_2025.xlsx', header=None)
df_2025.head()

,0,1,2,3,4,5,6,7,8,9,...,43,44,45,46,47,48,49,50,51,52
0,2025,Regular Unleaded Gasoline / Essence ordinaire,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,c/litre,"Retail prices, including taxes, self serve / P...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1/7,1/14,1/21,1/28,2/4,2/11,2/18,2/25,3/4,...,10/28,11/4,11/11,11/18,11/25,12/2,12/9,12/16,12/23,12/30
3,WHITEHORSE,179.9,179.9,179.9,179.9,181.9,184.9,184.9,184.9,184.9,...,154.9,151.9,149.9,149.9,149.9,149.9,149.9,149.9,149.9,149.9
4,VANCOUVER*,178.7,177.9,185.2,182.6,185.5,192.3,192.4,186.3,181.5,...,158.8,158.2,158.5,160.7,166,162.2,149.4,146.2,144.1,144.4


In [29]:
# get the entire date row( row 2, all columns from col 1 onwards )
print(df_2025.iloc[2,1:])

# Get the Toronto row (row 24, all columns from col 1 onward)
toronto_prices = df_2025.iloc[24, 1:]

print(toronto_prices)

1       1/7
2      1/14
3      1/21
4      1/28
5       2/4
6      2/11
7      2/18
8      2/25
9       3/4
10     3/11
11     3/18
12     3/25
13      4/1
14      4/8
15     4/15
16     4/22
17     4/29
18      5/6
19     5/13
20     5/20
21     5/27
22      6/3
23     6/10
24     6/17
25     6/24
26      7/1
27      7/8
28     7/15
29     7/22
30     7/29
31      8/5
32     8/12
33     8/19
34     8/26
35      9/2
36      9/9
37     9/16
38     9/23
39     9/30
40     10/7
41    10/14
42    10/21
43    10/28
44     11/4
45    11/11
46    11/18
47    11/25
48     12/2
49     12/9
50    12/16
51    12/23
52    12/30
Name: 2, dtype: object
1     154.7
2     156.7
3     158.1
4     155.3
5     155.5
6     156.9
7     157.3
8     155.1
9     152.3
10    151.5
11    152.1
12    154.6
13    153.2
14    135.2
15    128.2
16    132.1
17    136.7
18    135.3
19    135.4
20    138.9
21    137.9
22      135
23    134.1
24    136.6
25    141.5
26    134.7
27    133.8
28    137.3
29    136.3
30   

In [32]:
# Building a proper date string

year = '2025'
raw_dates = '1/7'
month, day = raw_dates.split('/')
date_str =f'{year}-{int(month):02d}-{int(day):02d}'

print(date_str)

2025-01-07


In [ ]:
# The mapping of city names in the dataset to more standardized city names

ONTARIO_CITIES = {
    'CITY OF TORONTO'  : 'Toronto',
    'TORONTO'          : 'Toronto',
    'BRAMPTON'         : 'Brampton',
    'ETOBICOKE'        : 'Etobicoke',
    'MISSISSAUGA'      : 'Mississauga',
    'NORTH YORK'       : 'North York',
    'SCARBOROUGH'      : 'Scarborough',
    'VAUGHAN'          : 'Vaughan / Markham',
    'MARKHAM'          : 'Vaughan / Markham',
    'OTTAWA'           : 'Ottawa',
    'KINGSTON'         : 'Kingston',
    'PETERBOROUGH'     : 'Peterborough',
    'WINDSOR'          : 'Windsor',
    'LONDON'           : 'London',
    'SUDBURY'          : 'Sudbury',
    'SAULT STE'        : 'Sault Ste. Marie',
    'THUNDER BAY'      : 'Thunder Bay',
    'NORTH BAY'        : 'North Bay',
    'TIMMINS'          : 'Timmins',
    'HAMILTON'         : 'Hamilton',
    'ST. CATHARINES'   : 'St. Catharines',
    'BARRIE'           : 'Barrie',
    'BRANTFORD'        : 'Brantford',
    'GUELPH'           : 'Guelph',
    'KITCHENER'        : 'Kitchener',
    'OSHAWA'           : 'Oshawa',
    'SARNIA'           : 'Sarnia',
    'ONTARIO AVE'      : 'Ontario Average',
    'ONTARIO AVERAGE'  : 'Ontario Average',
}





In [57]:
def parse_single_year(filepath):          # ← no more year parameter
    
    # extract year from filename automatically
    
    filename = os.path.basename(filepath)
    year     = filename.split('_')[-1].split('.')[0]
    
    df = pd.read_excel(filepath, header=None)
    
    # detect layout 2016
    
    sample = str(df.iloc[0, 1]).strip()
    if '/' in sample and len(sample) <= 5:
        # 2016: dates in row 0, cities from row 1
        date_row  = 0
        city_start = 1
    else:
        # 2017-2026: dates in row 2, cities from row 3
        date_row  = 2
        city_start = 3
    
    # find city rows
    
    targets = []
    for i in range(3, df.shape[0]):
        cell = str(df.iloc[i, 0]).upper().strip()
        for key, canonical_name in ONTARIO_CITIES.items():
            if key in cell:
                targets.append((i, canonical_name))
                break
    
    # build records
    
    records = []
    for col in range(1, df.shape[1]):
        raw = df.iloc[2, col]
        if pd.isna(raw):
            continue
        try:
            month, day = str(raw).strip().split('/')
            date_str = f'{year}-{int(month):02d}-{int(day):02d}'
        except:
            continue
        
         # use row_idx instead of city_row
        
        for row_idx, city_name in targets: 
            val = df.iloc[row_idx, col]
            if pd.notna(val):
                try:
                    records.append({
                        'date'      : date_str,
                        'city'      : city_name,
                        'price_cpl' : float(val)
                    })
                except ValueError:
                    pass
    
    return pd.DataFrame(records)

test = parse_single_year(r'C:\Users\t-zai\OneDrive\Desktop\PYTHON\gas_price\data\Unleaded_Retail (Incl. Tax)_WEEKLY_2025.xlsx')
print(test.shape)
test.head()

(1352, 3)


,date,city,price_cpl
0,2025-01-07,Toronto,154.7
1,2025-01-07,Brampton,155.1
2,2025-01-07,Etobicoke,155.0
3,2025-01-07,Mississauga,153.7
4,2025-01-07,North York,155.5


In [55]:
# process all files and combine into one dataframe

wide = result.pivot_table(
    index   = 'date',
    columns = 'city',
    values  = 'price_cpl',
    aggfunc = 'first'
)
wide.columns.name = None
wide = wide.reset_index()
wide.head()

,date,Barrie,Brampton,Brantford,Etobicoke,Guelph,Hamilton,Kingston,Kitchener,London,...,Sarnia,Sault Ste. Marie,Scarborough,St. Catharines,Sudbury,Thunder Bay,Timmins,Toronto,Vaughan / Markham,Windsor
0,2025-01-07,154.2,155.1,151.0,155.0,154.1,153.0,144.4,153.4,154.9,...,144.4,148.7,155.0,152.7,143.6,154.9,154.7,154.7,154.5,153.9
1,2025-01-14,155.7,156.4,152.8,156.0,155.6,154.5,145.9,154.9,156.4,...,149.4,152.0,156.2,153.9,151.4,151.6,154.9,156.7,156.5,155.5
2,2025-01-21,157.1,158.0,153.4,157.5,156.9,156.3,147.6,156.1,157.8,...,149.0,150.9,157.7,155.7,156.4,160.8,162.6,158.1,157.8,156.9
3,2025-01-28,154.3,155.2,152.9,155.0,154.2,153.5,147.6,153.2,155.1,...,151.0,150.1,154.8,153.0,155.1,156.5,166.9,155.3,155.2,154.5
4,2025-02-04,154.3,155.2,151.8,155.0,154.0,153.0,143.7,152.2,155.0,...,143.8,149.4,154.9,152.6,156.8,152.6,164.5,155.5,155.2,154.4


In [51]:
# Extracting the year from the filename automatically  
def extract_year_from_filename(filename):
    basename = os.path.basename(filename)
    parts = basename.split('_')
    for part in parts:
        if part.isdigit() and len(part) == 4:
            return part
    return None
